# Book Market Analytics – Books to Scrape
**End-to-end data science project**

This notebook scrapes the Books to Scrape practice catalogue, cleans the data, explores pricing patterns, trains a Linear Regression model, evaluates it against a baseline, and generates a results-based report.

**Important:** Run the notebook from top to bottom while connected to the internet. The raw and cleaned CSV files and report are generated from the data actually collected. Do not report results before running the notebook successfully.

## 1. Project objectives

1. Collect book title, price in GBP, star rating, stock availability and quantity where shown.
2. Collect genre and description where available to support additional analysis.
3. Clean and validate the dataset.
4. Visualise prices, ratings, genres and availability.
5. Train a Linear Regression model and compare it with a mean-price baseline.
6. Save the dataset, charts and an automatically populated project report.

## 2. Install and import libraries
Run the install cell only if these packages are missing from your environment.

In [ ]:
%pip install requests beautifulsoup4 pandas numpy matplotlib seaborn scikit-learn

In [1]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
OUTPUT_DIR = Path("book_market_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

BASE_URL = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; EducationalBookAnalytics/1.0)"}
session = requests.Session()
session.headers.update(HEADERS)

print("Output folder:", OUTPUT_DIR.resolve())

Output folder: C:\Users\User\Downloads\Book_Market_Analytics_Project\Book_Market_Analytics_Project\book_market_outputs


## 3. Helper functions for scraping

In [ ]:
def get_soup(url, retries=3, delay=0.5):
    """Download a page with basic retry handling and parse it with BeautifulSoup."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session.get(url, timeout=25)
            response.raise_for_status()
            return BeautifulSoup(response.text, "html.parser")
        except requests.RequestException as error:
            last_error = error
            if attempt < retries - 1:
                time.sleep(delay * (attempt + 1))
    raise last_error


def parse_book_detail(detail_url):
    """Return genre and description from a book detail page when available."""
    genre = "Unknown"
    description = ""
    try:
        soup = get_soup(detail_url)
        breadcrumbs = soup.select("ul.breadcrumb li a")
        # Breadcrumb order: Home, Books, Genre, Book title
        if len(breadcrumbs) >= 3:
            genre = breadcrumbs[2].get_text(" ", strip=True)

        description_heading = soup.find(
            lambda tag: tag.name in ("h2", "h3")
            and "product description" in tag.get_text(" ", strip=True).lower()
        )
        if description_heading:
            paragraph = description_heading.find_next("p")
            if paragraph:
                description = paragraph.get_text(" ", strip=True)
    except requests.RequestException as error:
        print(f"Could not read detail page {detail_url}: {error}")

    return genre, description

## 4. Data acquisition

The catalogue normally contains 50 pages with 20 books per page. This scraper follows the catalogue pages and visits each book's detail page to obtain genre and description where available. Requests are spaced out to avoid sending rapid repeated requests.

The code records failures so they can be checked and retried. A run is only considered complete if the collected row count is verified.

In [ ]:
records = []
failed_listing_pages = []
failed_detail_pages = []
total_pages = 50

for page_number in range(1, total_pages + 1):
    listing_url = urljoin(BASE_URL, f"catalogue/page-{page_number}.html")

    try:
        soup = get_soup(listing_url)
    except requests.RequestException as error:
        print(f"Listing page {page_number} failed: {error}")
        failed_listing_pages.append(page_number)
        continue

    book_cards = soup.select("article.product_pod")

    for card in book_cards:
        try:
            link = card.select_one("h3 a")
            price_element = card.select_one(".price_color")
            rating_element = card.select_one("p.star-rating")
            availability_element = card.select_one(".availability")

            if not all([link, price_element, rating_element, availability_element]):
                continue

            title = link.get("title", link.get_text(" ", strip=True)).strip()
            price_text = price_element.get_text(" ", strip=True)
            price_match = re.search(r"\d+(?:\.\d+)?", price_text)
            price_gbp = float(price_match.group()) if price_match else np.nan

            rating_classes = rating_element.get("class", [])
            rating_word = next(
                (item for item in rating_classes if item != "star-rating"),
                "Unknown"
            )

            stock_text = availability_element.get_text(" ", strip=True)
            availability = "In stock" if "in stock" in stock_text.lower() else "Out of stock"
            quantity_match = re.search(r"\((\d+)\s+available\)", stock_text, re.I)
            stock_quantity = int(quantity_match.group(1)) if quantity_match else np.nan

            detail_url = urljoin(listing_url, link.get("href", ""))
            genre, description = parse_book_detail(detail_url)

            records.append({
                "title": title,
                "price_gbp": price_gbp,
                "rating": rating_word,
                "availability": availability,
                "stock_quantity": stock_quantity,
                "genre": genre,
                "description": description,
                "detail_url": detail_url
            })

            time.sleep(0.08)

        except (AttributeError, TypeError, ValueError) as error:
            print(f"Skipped a listing on page {page_number}: {error}")

    print(f"Page {page_number}/{total_pages}: {len(book_cards)} listings; total collected: {len(records)}")
    time.sleep(0.2)

raw_df = pd.DataFrame(records)
raw_path = OUTPUT_DIR / "books_raw.csv"
raw_df.to_csv(raw_path, index=False)

print("\nRaw rows collected:", len(raw_df))
print("Unique titles:", raw_df["title"].nunique() if not raw_df.empty else 0)
print("Failed listing pages:", failed_listing_pages)
print("Raw data saved to:", raw_path.resolve())

if raw_df.empty:
    raise RuntimeError("No books were collected. Check your internet connection and try again.")

## 5. Inspect the raw data

In [ ]:
display(raw_df.head())
print("Shape:", raw_df.shape)
print("\nColumn types:")
display(raw_df.dtypes.to_frame("data_type"))
print("\nMissing values:")
display(raw_df.isna().sum().to_frame("missing_count"))
print("\nDuplicate rows:", raw_df.duplicated().sum())

## 6. Data cleaning and preparation

In [ ]:
clean_df = raw_df.copy()

clean_df["title"] = clean_df["title"].astype("string").str.strip()
clean_df["price_gbp"] = pd.to_numeric(clean_df["price_gbp"], errors="coerce")
clean_df["stock_quantity"] = pd.to_numeric(clean_df["stock_quantity"], errors="coerce")

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
clean_df["rating_score"] = clean_df["rating"].map(rating_map)

clean_df["genre"] = (
    clean_df["genre"].astype("string").str.strip()
    .replace({"": pd.NA, "Unknown": pd.NA})
    .fillna("Unknown")
)
clean_df["description"] = clean_df["description"].fillna("").astype(str).str.strip()
clean_df["availability"] = clean_df["availability"].fillna("Unknown").astype(str).str.strip()
clean_df["title_length"] = clean_df["title"].str.len()

rows_before = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=["title"], keep="first")
clean_df = clean_df.dropna(subset=["title", "price_gbp", "rating_score"])
clean_df = clean_df[clean_df["price_gbp"] > 0].copy()
rows_after = len(clean_df)

clean_path = OUTPUT_DIR / "books_cleaned.csv"
clean_df.to_csv(clean_path, index=False)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows removed:", rows_before - rows_after)
print("Unique titles remaining:", clean_df["title"].nunique())
print("Cleaned dataset saved to:", clean_path.resolve())
display(clean_df.head())

### Cleaning decisions

- Prices and stock quantities are stored as numeric values.
- Star words are preserved in `rating` and converted to a numeric `rating_score` for analysis.
- Duplicate titles are removed because the catalogue is expected to list each book once.
- Rows with missing titles, prices or ratings are removed from modelling.
- Missing stock quantities are not changed to zero because absence of a quantity does not prove that the book is out of stock.
- Missing genres are labelled `Unknown`; no genre is guessed.
- Descriptions are stripped of extra whitespace and retained as text for possible future analysis.

## 7. Descriptive statistics

In [ ]:
print("Number of cleaned books:", len(clean_df))
print("Number of genres including Unknown:", clean_df["genre"].nunique())
print("Mean price (£):", round(clean_df["price_gbp"].mean(), 2))
print("Median price (£):", round(clean_df["price_gbp"].median(), 2))
print("Minimum price (£):", round(clean_df["price_gbp"].min(), 2))
print("Maximum price (£):", round(clean_df["price_gbp"].max(), 2))
display(clean_df[["price_gbp", "rating_score", "stock_quantity", "title_length"]].describe().round(2))

## 8. Visualisation and interpretation

In [ ]:
# Chart 1: Price distribution
plt.figure(figsize=(9, 5))
sns.histplot(data=clean_df, x="price_gbp", bins=25, kde=True)
plt.title("Distribution of Book Prices")
plt.xlabel("Price (£)")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_price_distribution.png", dpi=150)
plt.show()

print("Interpretation prompt: Describe the main price range, skewness and any unusually expensive books based on this chart.")

In [ ]:
# Chart 2: Average price by rating
rating_summary = clean_df.groupby("rating_score")["price_gbp"].agg(["count", "mean", "median"])
display(rating_summary.round(2))

plt.figure(figsize=(8, 5))
sns.barplot(data=clean_df, x="rating_score", y="price_gbp", errorbar=None)
plt.title("Average Book Price by Star Rating")
plt.xlabel("Star Rating")
plt.ylabel("Average Price (£)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_price_by_rating.png", dpi=150)
plt.show()

print("Interpretation prompt: Compare average prices across ratings. Do not assume that a higher rating causes a higher price.")

In [ ]:
# Chart 3: Average price by genre, showing genres with at least five books
genre_summary = clean_df.groupby("genre")["price_gbp"].agg(["count", "mean", "median"])
genre_plot_data = genre_summary[genre_summary["count"] >= 5].sort_values("mean", ascending=False)

display(genre_plot_data.round(2))

if not genre_plot_data.empty:
    plt.figure(figsize=(11, max(5, len(genre_plot_data) * 0.3)))
    sns.barplot(data=genre_plot_data.reset_index(), x="mean", y="genre")
    plt.title("Average Book Price by Genre (minimum 5 books)")
    plt.xlabel("Average Price (£)")
    plt.ylabel("Genre")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "03_price_by_genre.png", dpi=150)
    plt.show()
else:
    print("Not enough genre groups with five or more books to plot.")

In [ ]:
# Chart 4: Price distribution by rating
plt.figure(figsize=(9, 5))
sns.boxplot(data=clean_df, x="rating_score", y="price_gbp")
plt.title("Book Price Distribution by Rating")
plt.xlabel("Star Rating")
plt.ylabel("Price (£)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_price_boxplot_by_rating.png", dpi=150)
plt.show()

In [ ]:
# Chart 5: Stock availability counts
availability_counts = clean_df["availability"].value_counts(dropna=False)
display(availability_counts.to_frame("book_count"))

plt.figure(figsize=(7, 5))
sns.countplot(data=clean_df, x="availability")
plt.title("Book Stock Availability")
plt.xlabel("Availability")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_stock_availability.png", dpi=150)
plt.show()

In [ ]:
# Chart 6: Correlation between numeric variables
numeric_columns = ["price_gbp", "rating_score", "title_length", "stock_quantity"]
correlation = clean_df[numeric_columns].corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numerical Variables")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_correlation_heatmap.png", dpi=150)
plt.show()

display(correlation.round(2))
print("Interpretation: Correlation describes association, not causation. Pairwise calculations use available values.")

## 9. Predictive modelling: Linear Regression

The target variable is `price_gbp`. The features are rating score, genre and title length. Genre is categorical and is one-hot encoded. The dataset is split into 80% training data and 20% test data. A fixed random seed makes the split reproducible.

The model is also compared with a mean-price baseline. This helps determine whether the features improve predictions compared with always predicting the average training price.

In [ ]:
features = ["rating_score", "genre", "title_length"]
target = "price_gbp"

model_df = clean_df[features + [target]].dropna().copy()
X = model_df[features]
y = model_df[target]

if len(model_df) < 10:
    raise ValueError("Too few usable rows for a meaningful train/test split.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("genre", OneHotEncoder(handle_unknown="ignore"), ["genre"])
    ],
    remainder="passthrough"
)

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])

linear_model.fit(X_train, y_train)
predictions = linear_model.predict(X_test)

baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(np.zeros((len(y_train), 1)), y_train)
baseline_predictions = baseline_model.predict(np.zeros((len(y_test), 1)))

def calculate_metrics(actual, predicted):
    return {
        "MAE (£)": mean_absolute_error(actual, predicted),
        "MSE (£ squared)": mean_squared_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted)
    }

metrics = pd.DataFrame({
    "Linear Regression": calculate_metrics(y_test, predictions),
    "Mean-price baseline": calculate_metrics(y_test, baseline_predictions)
}).T

display(metrics.round(3))

In [ ]:
# Actual versus predicted prices
results = pd.DataFrame({
    "actual_price_gbp": y_test.to_numpy(),
    "predicted_price_gbp": predictions
})
results["absolute_error_gbp"] = (
    results["actual_price_gbp"] - results["predicted_price_gbp"]
).abs()
results.to_csv(OUTPUT_DIR / "model_predictions.csv", index=False)

low = min(results["actual_price_gbp"].min(), results["predicted_price_gbp"].min())
high = max(results["actual_price_gbp"].max(), results["predicted_price_gbp"].max())

plt.figure(figsize=(7, 6))
sns.scatterplot(data=results, x="actual_price_gbp", y="predicted_price_gbp")
plt.plot([low, high], [low, high], linestyle="--")
plt.title("Actual vs Predicted Book Prices")
plt.xlabel("Actual Price (£)")
plt.ylabel("Predicted Price (£)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "07_actual_vs_predicted.png", dpi=150)
plt.show()

display(results.head(10).round(2))

In [ ]:
# Try a hypothetical new book. The genre should match a genre seen in the dataset where possible.
new_book = pd.DataFrame([{
    "rating_score": 4,
    "genre": "Mystery",
    "title_length": 25
}])

estimated_price = float(linear_model.predict(new_book)[0])
print(f"Estimated price: £{max(0, estimated_price):.2f}")
print("This is an estimate, not a guaranteed or commercially validated price.")

## 10. Automatically generate a report using the actual results

In [ ]:
# Calculate summaries used in the report
mean_price = clean_df["price_gbp"].mean()
median_price = clean_df["price_gbp"].median()
min_price = clean_df["price_gbp"].min()
max_price = clean_df["price_gbp"].max()

rating_means = clean_df.groupby("rating_score")["price_gbp"].mean()
rating_highest = int(rating_means.idxmax()) if not rating_means.empty else None
rating_lowest = int(rating_means.idxmin()) if not rating_means.empty else None

valid_genres = genre_summary[genre_summary["count"] >= 5]
highest_genre = valid_genres["mean"].idxmax() if not valid_genres.empty else "Not enough data"
lowest_genre = valid_genres["mean"].idxmin() if not valid_genres.empty else "Not enough data"

linear_mae = metrics.loc["Linear Regression", "MAE (£)"]
baseline_mae = metrics.loc["Mean-price baseline", "MAE (£)"]
linear_rmse = metrics.loc["Linear Regression", "RMSE (£)"]
linear_r2 = metrics.loc["Linear Regression", "R²"]
baseline_r2 = metrics.loc["Mean-price baseline", "R²"]

if linear_mae < baseline_mae:
    model_comparison = "Linear Regression had a lower MAE than the mean-price baseline on this test split."
else:
    model_comparison = "Linear Regression did not improve on the mean-price baseline's MAE on this test split."

report = f"""
# Book Market Analytics – Books to Scrape

## 1. Executive Summary
This project used Python, Requests and BeautifulSoup to collect book catalogue data from Books to Scrape. The cleaned dataset contained **{len(clean_df)} books** across **{clean_df['genre'].nunique()} genre labels**. The mean listed price was **£{mean_price:.2f}**, and the median was **£{median_price:.2f}**.

A Linear Regression model was trained to estimate price from rating score, genre and title length. Its test MAE was **£{linear_mae:.2f}**, RMSE was **£{linear_rmse:.2f}**, and R² was **{linear_r2:.3f}**. {model_comparison}

## 2. Objectives
- Scrape book title, price, rating and stock information.
- Clean and structure the collected records.
- Explore price, rating, genre and stock patterns.
- Train and evaluate a Linear Regression model.
- Develop cautious recommendations from the observed data.

## 3. Methodology
Listing pages were retrieved with Requests and parsed using BeautifulSoup. The scraper collected title, price in GBP, star rating, stock availability, quantity where displayed, genre and description where available. Prices and quantities were converted to numeric types. Star ratings were retained as words and mapped to numeric scores. Duplicate titles and records missing key modelling fields were removed. The cleaned data was exported to CSV.

The analysis used histograms, bar charts, box plots, a stock count plot and a correlation heatmap. The model used rating score, genre and title length, with one-hot encoding for genre. An 80:20 train-test split with random state 42 was used.

## 4. Findings
### 4.1 Price distribution
- Mean listed price: **£{mean_price:.2f}**
- Median listed price: **£{median_price:.2f}**
- Minimum listed price: **£{min_price:.2f}**
- Maximum listed price: **£{max_price:.2f}**

Compare the mean and median and refer to the saved price distribution chart to describe the observed shape. Avoid inferring why prices differ without supporting data.

### 4.2 Rating and price
The highest average price among rating groups was observed for rating **{rating_highest}**. The lowest average price among rating groups was observed for rating **{rating_lowest}**. This is a descriptive comparison and does not demonstrate that ratings cause prices to change.

### 4.3 Genre
Among genres with at least five books, the highest mean price was in **{highest_genre}**, while the lowest mean price was in **{lowest_genre}**. Genre comparisons should be interpreted with group sizes in mind.

### 4.4 Availability
{availability_counts.to_dict()}

Availability represents the state shown on the scraped pages at collection time, not sales or customer demand.

## 5. Predictive Model
| Metric | Linear Regression | Mean-price baseline |
|---|---:|---:|
| MAE (£) | {metrics.loc['Linear Regression', 'MAE (£)']:.3f} | {metrics.loc['Mean-price baseline', 'MAE (£)']:.3f} |
| MSE (£ squared) | {metrics.loc['Linear Regression', 'MSE (£ squared)']:.3f} | {metrics.loc['Mean-price baseline', 'MSE (£ squared)']:.3f} |
| RMSE (£) | {metrics.loc['Linear Regression', 'RMSE (£)']:.3f} | {metrics.loc['Mean-price baseline', 'RMSE (£)']:.3f} |
| R² | {metrics.loc['Linear Regression', 'R²']:.3f} | {metrics.loc['Mean-price baseline', 'R²']:.3f} |

MAE expresses the average absolute prediction error in pounds. RMSE is also in pounds and penalises larger errors more strongly. R² compares the model with a mean-based benchmark. The results should be interpreted together rather than using one metric alone.

## 6. Limitations
Books to Scrape is a practice website; its prices and ratings are randomly assigned and should not be treated as evidence of real commercial market behaviour. The catalogue does not provide actual sales, demand, publishing costs or profit margins. Stock quantities may be missing. A single random train-test split may vary from other splits, and the model's performance may be limited because the features contain little information about the assigned prices.

## 7. Recommendations
1. For retailers: use this workflow to monitor catalogue data, but validate pricing decisions against real sales, competitor prices and margins.
2. For publishers: investigate additional variables such as format, publisher, publication date, production costs and sales performance before making pricing decisions.
3. For analysts: compare alternative features and use cross-validation to test how stable model performance is.
4. For future work: repeat the analysis with a real retailer dataset before making commercial claims.

## 8. Conclusion
This project demonstrates an end-to-end data science workflow from scraping and cleaning through visual exploration and regression modelling. The results describe the collected practice catalogue only. The model should not be used to set real book prices without external validation.

## 9. References
- Books to Scrape. https://books.toscrape.com/
- Beautiful Soup Documentation. https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- Pandas Documentation. https://pandas.pydata.org/docs/
- Scikit-learn Documentation. https://scikit-learn.org/stable/
"""

report_path = OUTPUT_DIR / "Book_Market_Analytics_Report.md"
report_path.write_text(report, encoding="utf-8")
print("Report generated from this run:", report_path.resolve())
print(report[:2500])

## 11. Final validation and submission files
Check the printed count, failed pages, saved CSV files and generated report. If any listing pages failed, rerun the relevant pages before final submission. Include the notebook, `books_cleaned.csv`, and the generated report. The raw CSV and chart files are useful supporting evidence.

In [ ]:
print("Cleaned books:", len(clean_df))
print("Unique titles:", clean_df["title"].nunique())
print("Missing required values:")
display(clean_df[["title", "price_gbp", "rating", "availability"]].isna().sum())
print("\nFiles created:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)